In [2]:
# -Connect to online Vllm server
# vllm serve microsoft/Phi-3.5-vision-instruct --runner generate --allowed-local-media-path images/bowl_dining_table\
#   --trust-remote-code ---model-len 4096 --limit-mm-per-prompt.image 2 


In [4]:
import os
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

# Single-image input inference

# Public image URL for testing remote image processing
image_url = "https://vllm-public-assets.s3.us-west-2.amazonaws.com/vision_model_images/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"

# Create chat completion with remote image
chat_response = client.chat.completions.create(
    model="microsoft/Phi-3.5-vision-instruct",
    messages=[
        {
            "role": "user",
            "content": [
                # NOTE: The prompt formatting with the image token `<image>` is not needed
                # since the prompt will be processed automatically by the API server.
                {
                    "type": "text",
                    "text": "What’s in this image?",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_url},
                    "uuid": image_url,  # Optional
                },
            ],
        }
    ],
)
print("Chat completion output:", chat_response.choices[0].message.content)


In [ ]:
# Local image file path (update this to point to your actual image file)
image_file = "images/bowl_dining_table/COCO_train2014_000000386603.jpg"

# Create chat completion with local image file
# Launch the API server/engine with the --allowed-local-media-path argument.
if os.path.exists(image_file):
    chat_completion_from_local_image_url = client.chat.completions.create(
        model="microsoft/Phi-3.5-vision-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "What’s in this image?",
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"file://{image_file}"},
                    },
                ],
            }
        ],
    )
    result = chat_completion_from_local_image_url.choices[0].message.content
    print("Chat completion output from local image file:\n", result)
else:
    print(f"Local image file not found at {image_file}, skipping local file test.")



In [ ]:
# Multi-image input inference
image_url_duck = "https://vllm-public-assets.s3.us-west-2.amazonaws.com/multimodal_asset/duck.jpg"
image_url_lion = "https://vllm-public-assets.s3.us-west-2.amazonaws.com/multimodal_asset/lion.jpg"

chat_response = client.chat.completions.create(
    model="microsoft/Phi-3.5-vision-instruct",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What are the animals in these images?",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_url_duck},
                    "uuid": image_url_duck,  # Optional
                },
                {
                    "type": "image_url",
                    "image_url": {"url": image_url_lion},
                    "uuid": image_url_lion,  # Optional
                },
            ],
        }
    ],
)
print("Chat completion output:", chat_response.choices[0].message.content)

In [ ]:
image_file1 = "images/bowl_dining_table/COCO_train2014_000000386603.jpg"
image_file2 = "images/bowl_dining_table/COCO_train2014_000000132002.jpg"

if os.path.exists(image_file):
    chat_completion_from_local_image_url = client.chat.completions.create(
        model="microsoft/Phi-3.5-vision-instruct",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Describe each image separately",
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"file://{image_file1}"},
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"file://{image_file2}"},
                    },
                ],
            }
        ],
    )
    result = chat_completion_from_local_image_url.choices[0].message.content
    print("Chat completion output from local image file:\n", result)
else:
    print(f"Local image file not found at {image_file}, skipping local file test.")
